In [3]:
"""
Algorithm 3: First-order re-optimization for fixed topology.

Operates in the (lambda, s, tau) space of Reformulation II (P-HC),
which is the same space as Algorithm 1's station subproblems (SP-j).

At fixed station openings x_j in {0,1} and routed-demand forecasts
lambda_bar_j, the network-level problem decouples into J independent
station-level problems:

  min  kappa_j * s_j  +  v_j * phi(tau_j)  -  r_j(lambda_j)
  s.t. 0 <= lambda_j <= lambda_bar_j
       lambda_j^2 <= tau_j * s_j              (rotated cone)
       lambda_j + eps * x_j <= sqrt(s_j)      (stability margin)
       0 <= s_j <= M_j^2 * x_j               (capacity bound)
       tau_j in [0, tau_bar]

where phi(tau) = tau / (1 - sqrt(tau)) is the transformed M/M/1 queue
function (convex on [0,1), Proposition 3.2).

Algorithm 3 solves all J station problems simultaneously by:
  1. Computing the gradient of the smooth objective.
  2. Taking a gradient or Nesterov accelerated gradient step.
  3. Projecting back onto the feasible set (a per-station convex QP).
  4. Checking stationarity / objective-improvement stopping criterion.

Revenue model: linear in admitted demand, r_j(lambda_j) = p_j * lambda_j.
This is the correct regime for Reformulation II (unlike Reformulation I /
Algorithm 2, which suffers the revenue trap of Proposition 3.7).
"""

import numpy as np
import cvxpy as cp
import time


# ---------------------------------------------------------------------------
# Objective and gradient
# ---------------------------------------------------------------------------

def phi(tau: np.ndarray, clip: float = 0.999) -> np.ndarray:
    """Transformed M/M/1 queue function phi(tau) = tau / (1 - sqrt(tau))."""
    tau = np.clip(tau, 0.0, clip)
    return tau / (1.0 - np.sqrt(tau))


def dphi(tau: np.ndarray, clip: float = 0.999) -> np.ndarray:
    """Derivative phi'(tau) = (2 - sqrt(tau)) / (2*(1 - sqrt(tau))^2)."""
    tau = np.clip(tau, 1e-12, clip)
    sq = np.sqrt(tau)
    return (2.0 - sq) / (2.0 * (1.0 - sq) ** 2)


def objective_value(lam, s, tau, kappa_j, v_j, p_j):
    """Total objective across all J stations (scalar)."""
    return float(np.sum(kappa_j * s + v_j * phi(tau) - p_j * lam))


def gradient(lam, s, tau, kappa_j, v_j, p_j):
    """
    Gradients of the smooth objective w.r.t. (lambda, s, tau).

    d/d_lambda [ -p_j * lambda_j ]  = -p_j
    d/d_s      [ kappa_j * s_j   ]  = kappa_j
    d/d_tau    [ v_j * phi(tau_j)]  = v_j * phi'(tau_j)
    """
    grad_lam = -p_j
    grad_s   = kappa_j
    grad_tau = v_j * dphi(tau)
    return grad_lam, grad_s, grad_tau


# ---------------------------------------------------------------------------
# Projection onto the per-station feasible set
# ---------------------------------------------------------------------------

def project_feasible(lam_in, s_in, tau_in,
                     x_j, lam_bar_j, M_j,
                     epsilon=2.0, tau_bar=0.999):
    """
    Project (lam_in, s_in, tau_in) onto the feasible set of (P-HC) at
    fixed (x_j, lambda_bar_j):

        F_j = { (lambda, s, tau) :
                  0 <= lambda <= lambda_bar_j,
                  lambda^2 <= tau * s,
                  lambda + eps * x_j <= sqrt(s),
                  0 <= s <= M_j^2 * x_j,
                  0 <= tau <= tau_bar }

    Solved as a per-station convex QP (one small CVXPY problem per
    station). Stations where x_j=0 are trivially projected to (0,0,0).
    """
    J = len(x_j)
    lam_out = np.zeros(J)
    s_out   = np.zeros(J)
    tau_out = np.zeros(J)

    for j in range(J):
        if x_j[j] < 0.5:
            # Station closed: only feasible point is (0, 0, 0)
            continue

        lam_j = cp.Variable(nonneg=True)
        s_j   = cp.Variable(nonneg=True)
        tau_j = cp.Variable(nonneg=True)

        constraints = [
            lam_j <= float(lam_bar_j[j]),
            # lambda^2 <= tau * s  <=>  quad_over_lin(lambda, s) <= tau  (s > 0)
            cp.quad_over_lin(lam_j, s_j) <= tau_j,
            # lambda + eps <= sqrt(s)  <=>  (lambda + eps)^2 <= s  (both sides >= 0)
            cp.square(lam_j + epsilon * float(x_j[j])) <= s_j,
            s_j <= float(M_j[j] ** 2 * float(x_j[j])),
            tau_j <= tau_bar,
        ]

        # Euclidean projection: minimize squared distance to input point
        objective = cp.Minimize(
            cp.square(lam_j - float(lam_in[j]))
            + cp.square(s_j   - float(s_in[j]))
            + cp.square(tau_j - float(tau_in[j]))
        )

        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.CLARABEL, verbose=False)

        if prob.status in ("optimal", "optimal_inaccurate"):
            lam_out[j] = float(lam_j.value)
            s_out[j]   = float(s_j.value)
            tau_out[j] = float(tau_j.value)
        else:
            # Fallback: trivial feasible point (stability margin forces s >= eps^2)
            s_fall = (epsilon * float(x_j[j])) ** 2 + 1e-6
            lam_out[j] = 0.0
            s_out[j]   = min(s_fall, M_j[j] ** 2 * float(x_j[j]))
            tau_out[j] = 0.0

    return lam_out, s_out, tau_out


# ---------------------------------------------------------------------------
# Feasibility initialiser
# ---------------------------------------------------------------------------

def initial_feasible_point(x_j, lam_bar_j, M_j, epsilon=2.0, frac=0.5):
    """
    Construct a guaranteed-feasible starting point (lambda^0, s^0, tau^0).

    For each open station j, we set:
      lambda_j = frac * lambda_bar_j           (admit a fraction of routed demand)
      s_j      = (lambda_j + eps)^2 + delta    (just above the stability constraint)
      tau_j    = lambda_j^2 / s_j              (exactly on the rotated-cone boundary)

    This satisfies all constraints by construction.
    """
    J = len(x_j)
    lam0 = np.zeros(J)
    s0   = np.zeros(J)
    tau0 = np.zeros(J)

    for j in range(J):
        if x_j[j] < 0.5:
            continue
        lj = frac * lam_bar_j[j]
        sj = (lj + epsilon) ** 2 + 1e-4
        sj = min(sj, M_j[j] ** 2)
        tj = lj ** 2 / sj if sj > 0 else 0.0
        lam0[j] = lj
        s0[j]   = sj
        tau0[j] = tj

    return lam0, s0, tau0


# ---------------------------------------------------------------------------
# Main Algorithm 3
# ---------------------------------------------------------------------------

def first_order_reopt(x_j, lam_bar_j, kappa_j, v_j, M_j, p_j,
                      lam0=None, s0=None, tau0=None,
                      epsilon=2.0, tau_bar=0.999,
                      step_rule="backtracking",
                      alpha0=1.0, beta=0.5, c_armijo=1e-4,
                      accelerated=True,
                      eps_stat=1e-5, max_iter=500,
                      verbose=True):
    """
    Algorithm 3: First-order re-optimization for fixed topology.

    Parameters
    ----------
    x_j : array (J,)       Binary station openings (fixed, from Algorithm 1 or 2).
    lam_bar_j : array (J,) Routed-demand forecasts (updated by operator).
    kappa_j : array (J,)   Quadratic capacity-cost coefficients.
    v_j : array (J,)       Congestion penalty weights.
    M_j : array (J,)       Max service capacity per station.
    p_j : array (J,)       Linear revenue coefficient per station.
    lam0, s0, tau0 : arrays (J,) or None
        Initial feasible point. If None, computed automatically.
    epsilon : float         Stability margin.
    tau_bar : float         Max utilization proxy (< 1).
    step_rule : str         "backtracking" (Armijo line search) or "fixed".
    alpha0 : float          Initial / fixed step size.
    beta : float            Backtracking reduction factor.
    c_armijo : float        Armijo sufficient-decrease constant.
    accelerated : bool      If True, use Nesterov momentum (FISTA-style).
    eps_stat : float        Stopping tolerance on objective improvement.
    max_iter : int          Maximum iterations.
    verbose : bool          Print iteration log.

    Returns
    -------
    dict with final (lambda, s, tau), objective history, iteration count,
    runtime, and stationarity flag.
    """

    J = len(x_j)
    x_j = np.asarray(x_j, dtype=float)

    # ---- Initialise feasible point ---------------------------------------
    if lam0 is None or s0 is None or tau0 is None:
        lam0, s0, tau0 = initial_feasible_point(x_j, lam_bar_j, M_j, epsilon)

    lam = lam0.copy()
    s   = s0.copy()
    tau = tau0.copy()

    # Nesterov momentum variables (FISTA: y is the extrapolated point)
    y_lam, y_s, y_tau = lam.copy(), s.copy(), tau.copy()
    t_nesterov = 1.0

    obj_history = [objective_value(lam, s, tau, kappa_j, v_j, p_j)]
    converged = False
    t0_wall = time.time()

    if verbose:
        print(f"{'Iter':>5}  {'Objective':>14}  {'Improvement':>14}  {'Step':>10}")
        print("-" * 50)
        print(f"{'0':>5}  {obj_history[0]:>14.6f}  {'—':>14}  {'—':>10}")

    for t in range(1, max_iter + 1):

        # ---- Step 1: Gradient at current (or momentum) point ---------------
        if accelerated:
            gl, gs, gt = gradient(y_lam, y_s, y_tau, kappa_j, v_j, p_j)
        else:
            gl, gs, gt = gradient(lam, s, tau, kappa_j, v_j, p_j)

        # ---- Step 2: Gradient step + projection (with step-size rule) ------
        if step_rule == "backtracking":
            alpha = alpha0
            f_curr = objective_value(
                y_lam if accelerated else lam,
                y_s   if accelerated else s,
                y_tau if accelerated else tau,
                kappa_j, v_j, p_j
            )
            grad_sq_norm = np.sum(gl**2) + np.sum(gs**2) + np.sum(gt**2)

            for _ in range(50):   # max backtracking steps
                # Tentative gradient step
                lam_try = (y_lam if accelerated else lam) - alpha * gl
                s_try   = (y_s   if accelerated else s)   - alpha * gs
                tau_try = (y_tau if accelerated else tau)  - alpha * gt

                # Project onto feasible set
                lam_new, s_new, tau_new = project_feasible(
                    lam_try, s_try, tau_try,
                    x_j, lam_bar_j, M_j, epsilon, tau_bar
                )

                f_new = objective_value(lam_new, s_new, tau_new, kappa_j, v_j, p_j)

                # Armijo sufficient-decrease condition
                if f_new <= f_curr - c_armijo * alpha * grad_sq_norm:
                    break
                alpha *= beta

        else:  # fixed step size
            alpha = alpha0
            lam_try = (y_lam if accelerated else lam) - alpha * gl
            s_try   = (y_s   if accelerated else s)   - alpha * gs
            tau_try = (y_tau if accelerated else tau)  - alpha * gt
            lam_new, s_new, tau_new = project_feasible(
                lam_try, s_try, tau_try,
                x_j, lam_bar_j, M_j, epsilon, tau_bar
            )

        # ---- Nesterov momentum update (FISTA-style) -------------------------
        if accelerated:
            t_new = (1.0 + np.sqrt(1.0 + 4.0 * t_nesterov ** 2)) / 2.0
            momentum = (t_nesterov - 1.0) / t_new

            y_lam = lam_new + momentum * (lam_new - lam)
            y_s   = s_new   + momentum * (s_new   - s)
            y_tau = tau_new + momentum * (tau_new  - tau)

            # Project y back onto feasible set to keep it feasible
            y_lam, y_s, y_tau = project_feasible(
                y_lam, y_s, y_tau,
                x_j, lam_bar_j, M_j, epsilon, tau_bar
            )
            t_nesterov = t_new

        # ---- Step 4: Update iterates ----------------------------------------
        lam_prev, s_prev, tau_prev = lam.copy(), s.copy(), tau.copy()
        lam, s, tau = lam_new, s_new, tau_new

        obj = objective_value(lam, s, tau, kappa_j, v_j, p_j)
        improvement = obj_history[-1] - obj
        obj_history.append(obj)

        if verbose:
            print(f"{t:>5}  {obj:>14.6f}  {improvement:>14.8f}  {alpha:>10.6f}")

        # ---- Stopping criterion: objective improvement below eps_stat -------
        if abs(improvement) < eps_stat:
            converged = True
            if verbose:
                print(f"\nConverged at iteration {t}: improvement={improvement:.2e} < eps_stat={eps_stat:.2e}")
            break

    runtime = time.time() - t0_wall

    if verbose and not converged:
        print(f"\nReached max_iter={max_iter}. Final objective: {obj_history[-1]:.6f}")

    return {
        "lambda_j": lam,
        "s_j": s,
        "tau_j": tau,
        "mu_j": np.sqrt(np.maximum(s, 0.0)),           # recover mu from s = mu^2
        "rho_j": np.where(s > 1e-12,                  # recover rho = lambda/mu
                          lam / np.sqrt(np.maximum(s, 1e-12)), 0.0),
        "objective": obj_history[-1],
        "obj_history": obj_history,
        "iterations": t,
        "converged": converged,
        "runtime_sec": runtime,
        "step_size_final": alpha,
    }

In [4]:
"""
Test Algorithm 3 on the toy instance from the notebook.

Simulates the operator workflow:
  1. Run Algorithm 1 to get the optimal topology (x_j, y_ij).
  2. Receive an updated demand forecast.
  3. Run Algorithm 3 to re-optimise station-level controls
     (lambda, s, tau) without reopening the topology question.
  4. Verify: compare standard vs accelerated, and test a second
     demand update to show repeated re-optimisation.
"""

import numpy as np
import sys
sys.path.insert(0, '.')

# ---- Fixed topology from Algorithm 1 toy instance ----------------------
# Algorithm 1 found x_j = [1, 0] (only station 0 opens),
# with all demand routed there: lambda_bar = [15, 0].

x_j       = np.array([1.0, 0.0])
kappa_j   = np.array([40.0, 50.0])
v_j       = np.array([0.8,  0.7])
M_j       = np.array([40.0, 15.0])
p_j       = np.array([200.0, 200.0])
epsilon   = 2.0

# ---- Demand forecast 1: original ----------------------------------------
print("=" * 60)
print("DEMAND FORECAST 1 (original): lambda_bar = [15, 0]")
print("=" * 60)
lam_bar_1 = np.array([15.0, 0.0])

out1 = first_order_reopt(
    x_j, lam_bar_1, kappa_j, v_j, M_j, p_j,
    epsilon=epsilon,
    step_rule="backtracking",
    accelerated=True,
    eps_stat=1e-6,
    max_iter=200,
    verbose=True,
)

print(f"\nlambda_j : {out1['lambda_j']}")
print(f"mu_j     : {out1['mu_j']}")
print(f"rho_j    : {out1['rho_j']}")
print(f"tau_j    : {out1['tau_j']}")
print(f"objective: {out1['objective']:.6f}")
print(f"runtime  : {out1['runtime_sec']:.3f}s  ({out1['iterations']} iters)")

# ---- Demand forecast 2: higher demand -----------------------------------
print("\n" + "=" * 60)
print("DEMAND FORECAST 2 (demand spike): lambda_bar = [25, 0]")
print("=" * 60)
lam_bar_2 = np.array([25.0, 0.0])

out2 = first_order_reopt(
    x_j, lam_bar_2, kappa_j, v_j, M_j, p_j,
    epsilon=epsilon,
    step_rule="backtracking",
    accelerated=True,
    eps_stat=1e-6,
    max_iter=200,
    verbose=True,
)

print(f"\nlambda_j : {out2['lambda_j']}")
print(f"mu_j     : {out2['mu_j']}")
print(f"rho_j    : {out2['rho_j']}")
print(f"objective: {out2['objective']:.6f}")
print(f"runtime  : {out2['runtime_sec']:.3f}s  ({out2['iterations']} iters)")

# ---- Compare accelerated vs non-accelerated on forecast 2 ---------------
print("\n" + "=" * 60)
print("COMPARISON: accelerated vs standard gradient (forecast 2)")
print("=" * 60)

out_std = first_order_reopt(
    x_j, lam_bar_2, kappa_j, v_j, M_j, p_j,
    epsilon=epsilon,
    step_rule="backtracking",
    accelerated=False,
    eps_stat=1e-6,
    max_iter=500,
    verbose=False,
)

out_acc = first_order_reopt(
    x_j, lam_bar_2, kappa_j, v_j, M_j, p_j,
    epsilon=epsilon,
    step_rule="backtracking",
    accelerated=True,
    eps_stat=1e-6,
    max_iter=500,
    verbose=False,
)

print(f"{'':30s}  {'Standard':>12}  {'Accelerated':>12}")
print(f"{'Iterations to converge':30s}  {out_std['iterations']:>12}  {out_acc['iterations']:>12}")
print(f"{'Final objective':30s}  {out_std['objective']:>12.6f}  {out_acc['objective']:>12.6f}")
print(f"{'Runtime (s)':30s}  {out_std['runtime_sec']:>12.3f}  {out_acc['runtime_sec']:>12.3f}")
print(f"{'Converged':30s}  {str(out_std['converged']):>12}  {str(out_acc['converged']):>12}")

DEMAND FORECAST 1 (original): lambda_bar = [15, 0]
 Iter       Objective     Improvement        Step
--------------------------------------------------
    0     2112.372414               —           —
    1     1332.215207    780.15720683    1.000000
    2      712.699524    619.51568313    1.000000
    3      252.821707    459.87781633    1.000000
    4      151.164405    101.65730264    0.062500
    5      150.498947      0.66545791    1.000000
    6      150.957838     -0.45889102    0.250000
    7      151.212594     -0.25475585    0.015625
    8      151.267027     -0.05443382    0.015625
    9      151.177677      0.08935066    0.015625
   10      151.084697      0.09297978    0.003906
   11      151.003253      0.08144388    0.001953
   12      150.929851      0.07340172    0.001953
   13      150.862359      0.06749187    0.001953
   14      150.804546      0.05781296    0.000977
   15      150.753873      0.05067376    0.000977
   16      150.708604      0.04526886    0.00097